### **Laboratorio 4 - Chunking y dense retrieval con evaluación controlada**

#### **Pregunta experimental**

> ¿Cómo afecta el tamaño objetivo de los chunks a la recuperación de evidencia relevante cuando el corpus, las consultas, las relevancias, el overlap, el modelo de embeddings, la similitud, el índice y `top-k` permanecen fijos?.

La variable independiente es solamente:

```text
target_words
```

con tres niveles:

```text
small    = 100
baseline = 180
large    = 320
```


#### **0. Protocolo antes de ejecutar**

Completa antes de ver resultados:

```text
Pregunta:
¿Cómo afecta target_words al Recall@k?

Hipótesis:

Baseline:
target_words = 180

Modificación:
target_words in {100, 320}

Variables fijas:
corpus
queries
qrels
overlap_passages = 1
embedding model
normalización L2
similarity = inner product
index = IndexFlatIP
top-k in {1, 3, 5}

Métrica principal:
mean Recall@3
```

No modifiques benchmark, qrels o modelo después de observar resultados para fabricar un patrón deseado.

#### **1. Configuración**

La ejecución canónica usa un encoder real.

Para validar el notebook sin descargar modelos:

```bash
CC0F4_RUN_REAL_RETRIEVAL=0
```

El modo offline sirve únicamente para comprobar carga de datos, chunking, ranking, métricas y análisis. Sus números **no** son resultados del experimento canónico.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

SEED = 42
np.random.seed(SEED)

MODEL_ID = "intfloat/multilingual-e5-small"

RUN_REAL_RETRIEVAL = (
    os.environ.get("CC0F4_RUN_REAL_RETRIEVAL", "1") == "1"
)

TARGET_WORDS = {
    "small": 100,
    "baseline": 180,
    "large": 320,
}

OVERLAP_PASSAGES = 1
TOP_K_VALUES = [1, 3, 5]
MAX_K = max(TOP_K_VALUES)

print("MODEL_ID:", MODEL_ID)
print("RUN_REAL_RETRIEVAL:", RUN_REAL_RETRIEVAL)
print("TARGET_WORDS:", TARGET_WORDS)
print("OVERLAP_PASSAGES:", OVERLAP_PASSAGES)

#### **2. Benchmark**

El benchmark contiene:

```text
corpus_semana4.jsonl
queries_semana4.jsonl
qrels_semana4.json
```

La relevancia se define sobre `passage_id`, no sobre `chunk_id`. Así los qrels permanecen estables aunque cambie el chunking.

In [ ]:
def resolve_data_dir() -> Path:
    explicit = os.environ.get("CC0F4_SEMANA4_DATA")

    if explicit:
        path = Path(explicit).expanduser().resolve()

        if path.is_dir():
            return path

        raise FileNotFoundError(
            f"No existe CC0F4_SEMANA4_DATA={path}"
        )

    cwd = Path.cwd().resolve()
    candidates = []

    for root in [cwd, *cwd.parents]:
        candidates.extend(
            [
                root / "Semana4" / "datos",
                root / "datos",
            ]
        )

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate in seen:
            continue

        seen.add(candidate)

        if (
            (candidate / "corpus_semana4.jsonl").is_file()
            and (candidate / "queries_semana4.jsonl").is_file()
            and (candidate / "qrels_semana4.json").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la carpeta de datos de Semana 4. "
        "Define CC0F4_SEMANA4_DATA si ejecutas desde otra ubicación."
    )


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido en {path}, línea {line_number}"
                ) from exc

    return rows


DATA_DIR = resolve_data_dir()

documents = read_jsonl(
    DATA_DIR / "corpus_semana4.jsonl"
)

queries = read_jsonl(
    DATA_DIR / "queries_semana4.jsonl"
)

qrels = json.loads(
    (DATA_DIR / "qrels_semana4.json").read_text(
        encoding="utf-8"
    )
)

print("DATA_DIR:", DATA_DIR)
print("Documentos:", len(documents))
print("Consultas:", len(queries))
print("Qrels:", len(qrels))

In [ ]:
def validate_dataset(
    documents: list[dict[str, Any]],
    queries: list[dict[str, Any]],
    qrels: dict[str, list[str]],
) -> None:
    doc_ids = [
        doc["doc_id"]
        for doc in documents
    ]

    query_ids = [
        row["query_id"]
        for row in queries
    ]

    if len(doc_ids) != len(set(doc_ids)):
        raise ValueError("Hay doc_id duplicados.")

    if len(query_ids) != len(set(query_ids)):
        raise ValueError("Hay query_id duplicados.")

    passage_ids = set()

    for doc in documents:
        if not doc.get("passages"):
            raise ValueError(
                f"{doc['doc_id']} no contiene passages."
            )

        for passage in doc["passages"]:
            passage_id = passage["passage_id"]

            if passage_id in passage_ids:
                raise ValueError(
                    f"passage_id duplicado: {passage_id}"
                )

            passage_ids.add(passage_id)

    if set(query_ids) != set(qrels):
        raise ValueError(
            "Cada query_id debe tener exactamente una entrada en qrels."
        )

    missing_passages = {
        passage_id
        for relevant in qrels.values()
        for passage_id in relevant
        if passage_id not in passage_ids
    }

    if missing_passages:
        raise ValueError(
            "Qrels referencia passages inexistentes: "
            f"{sorted(missing_passages)}"
        )

    if any(
        len(relevant) == 0
        for relevant in qrels.values()
    ):
        raise ValueError(
            "Toda consulta debe tener al menos una evidencia relevante."
        )


validate_dataset(
    documents,
    queries,
    qrels,
)

all_passages = [
    passage
    for doc in documents
    for passage in doc["passages"]
]

print("Integridad del benchmark: OK")
print("Passages:", len(all_passages))

pd.DataFrame(
    queries
)["difficulty"].value_counts()

#### **3. Chunking preservando trazabilidad**

Cada documento contiene passages atómicos.

El chunker agrupa passages consecutivos hasta aproximarse a `target_words`.

Metadata conservada:

```text
chunk_id
doc_id
title
passage_ids
n_words
text
```

El overlap se mantiene fijo en **un passage**.

In [ ]:
def build_chunks(
    documents: list[dict[str, Any]],
    target_words: int,
    overlap_passages: int = 1,
) -> list[dict[str, Any]]:
    if target_words <= 0:
        raise ValueError(
            "target_words debe ser positivo."
        )

    if overlap_passages < 0:
        raise ValueError(
            "overlap_passages no puede ser negativo."
        )

    chunks = []

    for doc in documents:
        passages = doc["passages"]
        start = 0
        chunk_number = 0

        while start < len(passages):
            current = []
            current_words = 0
            end = start

            while end < len(passages):
                passage = passages[end]
                passage_words = len(
                    passage["text"].split()
                )

                if (
                    current
                    and current_words + passage_words > target_words
                ):
                    break

                current.append(passage)
                current_words += passage_words
                end += 1

            if not current:
                raise RuntimeError(
                    "El chunker no pudo avanzar."
                )

            chunk_number += 1

            chunks.append(
                {
                    "chunk_id": (
                        f"{doc['doc_id']}"
                        f"-C{chunk_number:02d}"
                        f"-W{target_words}"
                    ),
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "passage_ids": [
                        p["passage_id"]
                        for p in current
                    ],
                    "n_words": current_words,
                    "text": " ".join(
                        p["text"]
                        for p in current
                    ),
                }
            )

            if end >= len(passages):
                break

            next_start = end - overlap_passages

            if next_start <= start:
                next_start = start + 1

            start = next_start

    return chunks


chunk_sets = {
    name: build_chunks(
        documents,
        target_words=target_words,
        overlap_passages=OVERLAP_PASSAGES,
    )
    for name, target_words in TARGET_WORDS.items()
}

chunk_stats = []

for name, chunks in chunk_sets.items():
    n_words = np.array(
        [
            chunk["n_words"]
            for chunk in chunks
        ],
        dtype=float,
    )

    chunk_stats.append(
        {
            "condition": name,
            "target_words": TARGET_WORDS[name],
            "n_chunks": len(chunks),
            "mean_words": float(n_words.mean()),
            "median_words": float(np.median(n_words)),
            "max_words": int(n_words.max()),
        }
    )

chunk_stats_df = pd.DataFrame(
    chunk_stats
).sort_values("target_words")

chunk_stats_df


def validate_realized_overlap(
    chunks: list[dict[str, Any]],
    expected_overlap: int,
) -> None:
    by_document: dict[str, list[dict[str, Any]]] = {}

    for chunk in chunks:
        by_document.setdefault(
            chunk["doc_id"],
            [],
        ).append(chunk)

    for doc_id, doc_chunks in by_document.items():
        for previous, current in zip(
            doc_chunks,
            doc_chunks[1:],
        ):
            realized = len(
                set(previous["passage_ids"])
                & set(current["passage_ids"])
            )

            if realized != expected_overlap:
                raise AssertionError(
                    f"{doc_id}: overlap realizado={realized}, "
                    f"esperado={expected_overlap}."
                )


for condition, chunks in chunk_sets.items():
    validate_realized_overlap(
        chunks,
        expected_overlap=OVERLAP_PASSAGES,
    )

print("Overlap realizado entre chunks consecutivos: OK")


#### **Checkpoint antes de embeddings**

1. ¿Qué condición produce más chunks?,
2. ¿Cuál conserva más contexto dentro de cada vector?,
3. ¿Cuál podría diluir una evidencia muy específica?,
4. ¿Por qué el número de chunks no permite decidir por sí solo cuál condición es mejor?.

La respuesta debe venir de retrieval sobre consultas conocidas.

#### **4. Encoder fijo**

En ejecución real:

```text
chunks  -> SentenceTransformer -> embeddings L2-normalizados
queries -> SentenceTransformer -> embeddings L2-normalizados
```

El mismo modelo se utiliza en las tres condiciones.

El modelo E5 usa prefijos de tarea:

```text
query: <consulta>
passage: <chunk>
```

Antes de generar embeddings se verifica que ningún chunk exceda
`encoder.max_seq_length`. Si una condición introduce truncación, el
experimento se detiene porque eso añadiría una segunda variable.


In [ ]:
if RUN_REAL_RETRIEVAL:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(MODEL_ID)
else:
    encoder = None

    print(
        "MODO VALIDACION: TF-IDF + SVD. "
        "No interpretar estas métricas como dense retrieval real."
    )


def encode_condition(
    chunks: list[dict[str, Any]],
    query_rows: list[dict[str, Any]],
) -> tuple[np.ndarray, np.ndarray]:
    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    query_texts = [
        row["query"]
        for row in query_rows
    ]

    if RUN_REAL_RETRIEVAL:
        chunk_inputs = [
            f"passage: {text}"
            for text in chunk_texts
        ]

        query_inputs = [
            f"query: {text}"
            for text in query_texts
        ]

        token_lengths = [
            len(
                encoder.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=False,
                )["input_ids"]
            )
            for text in chunk_inputs
        ]

        max_tokens = max(
            token_lengths
        )

        if max_tokens > encoder.max_seq_length:
            raise ValueError(
                "Una condición excede la ventana del encoder: "
                f"max_tokens={max_tokens}, "
                f"limit={encoder.max_seq_length}. "
                "Esto confundiría chunking con truncación."
            )

        chunk_vectors = encoder.encode(
            chunk_inputs,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        query_vectors = encoder.encode(
            query_inputs,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        return chunk_vectors, query_vectors

    # Este branch valida el software, no el experimento canónico.
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True,
    )

    chunk_sparse = vectorizer.fit_transform(
        chunk_texts
    )

    query_sparse = vectorizer.transform(
        query_texts
    )

    n_components = min(
        64,
        chunk_sparse.shape[0] - 1,
        chunk_sparse.shape[1] - 1,
    )

    if n_components < 2:
        raise ValueError(
            "No hay dimensiones suficientes para SVD."
        )

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=SEED,
    )

    chunk_vectors = svd.fit_transform(
        chunk_sparse
    )

    query_vectors = svd.transform(
        query_sparse
    )

    return (
        normalize(
            chunk_vectors,
            norm="l2",
        ).astype("float32"),
        normalize(
            query_vectors,
            norm="l2",
        ).astype("float32"),
    )


#### **5. Índice exacto**

`IndexFlatIP` realiza búsqueda exhaustiva por inner product.

Con vectores normalizados:

```text
producto interno == similitud coseno
```

In [ ]:
def exact_top_k(
    scores: np.ndarray,
    k: int,
) -> tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[1])

    indices = np.argsort(
        -scores,
        axis=1,
    )[:, :k]

    values = np.take_along_axis(
        scores,
        indices,
        axis=1,
    )

    return values, indices


class NumpyFlatIP:
    def __init__(self, dimension: int):
        self.dimension = dimension
        self.vectors = None

    def add(
        self,
        vectors: np.ndarray,
    ) -> None:
        vectors = np.asarray(
            vectors,
            dtype="float32",
        )

        if (
            vectors.ndim != 2
            or vectors.shape[1] != self.dimension
        ):
            raise ValueError(
                "Dimensión incompatible."
            )

        self.vectors = vectors

    def search(
        self,
        query_vectors: np.ndarray,
        k: int,
    ) -> tuple[np.ndarray, np.ndarray]:
        if self.vectors is None:
            raise RuntimeError(
                "El índice está vacío."
            )

        queries = np.asarray(
            query_vectors,
            dtype="float32",
        )

        scores = (
            queries
            @ self.vectors.T
        )

        return exact_top_k(
            scores,
            k,
        )


def build_index(
    vectors: np.ndarray,
):
    if RUN_REAL_RETRIEVAL:
        import faiss

        index = faiss.IndexFlatIP(
            vectors.shape[1]
        )
    else:
        index = NumpyFlatIP(
            vectors.shape[1]
        )

    index.add(
        vectors.astype("float32")
    )

    return index

#### **6. Recall@k sobre evidencia**

Para una consulta:

$$
Recall@k
=
\frac{\text{evidencia gold cubierta por los top-k chunks}}
     {\text{evidencia gold total}}
$$

La definición se mantiene estable aunque cambie la estrategia de segmentación (chunking).

In [ ]:
def evaluate_retrieval(
    chunks: list[dict[str, Any]],
    query_rows: list[dict[str, Any]],
    qrels: dict[str, list[str]],
    scores: np.ndarray,
    indices: np.ndarray,
    k_values: list[int],
) -> tuple[pd.DataFrame, dict[str, float]]:
    rows = []

    for q_idx, query_row in enumerate(
        query_rows
    ):
        query_id = query_row["query_id"]
        gold = set(qrels[query_id])

        row = {
            "query_id": query_id,
            "query": query_row["query"],
            "difficulty": query_row["difficulty"],
        }

        for k in k_values:
            retrieved_indices = indices[
                q_idx,
                : min(k, indices.shape[1]),
            ]

            covered = set()

            for chunk_idx in retrieved_indices:
                covered.update(
                    chunks[int(chunk_idx)][
                        "passage_ids"
                    ]
                )

            recall = (
                len(gold & covered)
                / len(gold)
            )

            row[f"recall@{k}"] = float(
                recall
            )

        top_chunks = []

        for rank, chunk_idx in enumerate(
            indices[q_idx],
            start=1,
        ):
            chunk = chunks[
                int(chunk_idx)
            ]

            top_chunks.append(
                {
                    "rank": rank,
                    "chunk_id": chunk["chunk_id"],
                    "doc_id": chunk["doc_id"],
                    "passage_ids": chunk[
                        "passage_ids"
                    ],
                    "score": float(
                        scores[
                            q_idx,
                            rank - 1,
                        ]
                    ),
                    "text": chunk["text"],
                }
            )

        row["top_chunks"] = top_chunks
        rows.append(row)

    per_query = pd.DataFrame(rows)

    summary = {
        f"Recall@{k}": float(
            per_query[
                f"recall@{k}"
            ].mean()
        )
        for k in k_values
    }

    return per_query, summary

#### **7. Ejecutar las tres condiciones**

```text
misma evidencia
mismo encoder
misma métrica
mismo índice
mismo top-k

cambia solamente: target_words
```

In [ ]:
condition_results = {}
summary_rows = []

for condition, chunks in chunk_sets.items():
    chunk_vectors, query_vectors = encode_condition(
        chunks,
        queries,
    )

    assert chunk_vectors.ndim == 2
    assert query_vectors.ndim == 2
    assert (
        chunk_vectors.shape[1]
        == query_vectors.shape[1]
    )

    index = build_index(
        chunk_vectors
    )

    scores, indices = index.search(
        query_vectors.astype("float32"),
        MAX_K,
    )

    per_query, metrics = evaluate_retrieval(
        chunks=chunks,
        query_rows=queries,
        qrels=qrels,
        scores=scores,
        indices=indices,
        k_values=TOP_K_VALUES,
    )

    condition_results[
        condition
    ] = {
        "chunks": chunks,
        "per_query": per_query,
        "scores": scores,
        "indices": indices,
    }

    stats_row = chunk_stats_df[
        chunk_stats_df[
            "condition"
        ] == condition
    ].iloc[0]

    summary_rows.append(
        {
            "condition": condition,
            "target_words": TARGET_WORDS[
                condition
            ],
            "n_chunks": int(
                stats_row["n_chunks"]
            ),
            "mean_words": float(
                stats_row["mean_words"]
            ),
            **metrics,
        }
    )

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("target_words")
    .reset_index(drop=True)
)

summary_df

#### **8. Interpretar el resultado agregado**

Responde después de ejecutar:

1. ¿Qué condición obtuvo mayor `Recall@3`?
2. ¿La condición con más chunks fue necesariamente la mejor?
3. ¿El patrón es igual para `Recall@1`, `Recall@3` y `Recall@5`?
4. ¿Los datos justifican afirmar que un tamaño de chunk es universalmente mejor?.

```text
resultado en este benchmark != regla universal de chunking
```

#### **9. Análisis pareado por consulta**

Una métrica promedio puede ocultar consultas que cambian de comportamiento.

Comparamos `Recall@3` de cada variante contra el baseline.

In [ ]:
baseline_per_query = (
    condition_results["baseline"][
        "per_query"
    ][
        [
            "query_id",
            "query",
            "difficulty",
            "recall@3",
        ]
    ]
    .rename(
        columns={
            "recall@3": "baseline_recall@3",
        }
    )
)


def compare_with_baseline(
    condition: str,
) -> pd.DataFrame:
    variant = (
        condition_results[
            condition
        ]["per_query"][
            [
                "query_id",
                "recall@3",
            ]
        ]
        .rename(
            columns={
                "recall@3": (
                    f"{condition}_recall@3"
                ),
            }
        )
    )

    comparison = (
        baseline_per_query
        .merge(
            variant,
            on="query_id",
            how="inner",
            validate="one_to_one",
        )
    )

    comparison["delta"] = (
        comparison[
            f"{condition}_recall@3"
        ]
        - comparison[
            "baseline_recall@3"
        ]
    )

    return comparison.sort_values(
        [
            "delta",
            "query_id",
        ]
    )


small_comparison = compare_with_baseline(
    "small"
)

large_comparison = compare_with_baseline(
    "large"
)

print("small vs baseline")
display(small_comparison)

print()
print("large vs baseline")
display(large_comparison)

#### **10. Auditar un caso**

Selecciona una consulta con `delta != 0`.

No basta con decir que el modelo se confundió. Debes inspeccionar los chunks exactos recuperados.

In [ ]:
def show_query_audit(
    query_id: str,
    conditions: list[str] | None = None,
    top_k: int = 3,
) -> None:
    if conditions is None:
        conditions = [
            "small",
            "baseline",
            "large",
        ]

    query_row = next(
        row
        for row in queries
        if row["query_id"] == query_id
    )

    print("query_id:", query_id)
    print("query:", query_row["query"])
    print("gold:", qrels[query_id])
    print()

    for condition in conditions:
        per_query = (
            condition_results[
                condition
            ]["per_query"]
        )

        row = per_query[
            per_query[
                "query_id"
            ] == query_id
        ].iloc[0]

        print("=" * 70)
        print(
            "condition:",
            condition,
            "| target_words:",
            TARGET_WORDS[condition],
            "| Recall@3:",
            row["recall@3"],
        )

        for item in row[
            "top_chunks"
        ][:top_k]:
            print()
            print(
                f"rank={item['rank']} "
                f"score={item['score']:.4f} "
                f"chunk={item['chunk_id']}"
            )
            print(
                "passages:",
                item["passage_ids"],
            )
            print(
                item["text"][:450]
            )


changed_query_ids = sorted(
    set(
        small_comparison.loc[
            small_comparison[
                "delta"
            ] != 0,
            "query_id",
        ]
    )
    | set(
        large_comparison.loc[
            large_comparison[
                "delta"
            ] != 0,
            "query_id",
        ]
    )
)

if changed_query_ids:
    show_query_audit(
        changed_query_ids[0]
    )
else:
    print(
        "No hubo cambios en Recall@3. "
        "Audita una consulta ambiguous y discute por qué."
    )

#### **11. Análisis por dificultad**

Las etiquetas `clear` y `ambiguous` son una ayuda didáctica, no una propiedad universal de una consulta.

Se usan para observar si el comportamiento agregado cambia entre ambos grupos.

In [ ]:
difficulty_rows = []

for condition, result in condition_results.items():
    grouped = (
        result["per_query"]
        .groupby("difficulty")[
            [
                "recall@1",
                "recall@3",
                "recall@5",
            ]
        ]
        .mean()
        .reset_index()
    )

    grouped.insert(
        0,
        "condition",
        condition,
    )

    difficulty_rows.append(
        grouped
    )

difficulty_df = pd.concat(
    difficulty_rows,
    ignore_index=True,
)

difficulty_df

#### **12. Conclusión técnica**

Completa una conclusión de 6 a 10 líneas:

```text
Pregunta:
Hipótesis previa:
Baseline:
Modificación:
Métrica principal:
Resultado observado:
Un error o caso particular:
Limitación:
Conclusión:
```

Forma correcta:

> Bajo este corpus, estas consultas, estos qrels y este encoder, modificar `target_words` produjo...

Forma incorrecta:

> Los chunks de X palabras siempre son mejores.

#### **13. Preguntas adicionales**

1. ¿Por qué los qrels se definen sobre passages y no sobre `chunk_id`?
2. ¿Qué ocurriría si cambiáramos simultáneamente `target_words` y el modelo de embeddings?
3. ¿Por qué `IndexFlatIP` no permite concluir nada sobre el trade-off de ANN?
4. ¿Qué diferencia existe entre mejorar `Recall@5` y mejorar la calidad final de un sistema RAG?
5. ¿Por qué aumentar `top-k` puede aumentar cobertura y también introducir ruido?
6. ¿Qué información adicional necesitaríamos para elegir chunking en producción?.

#### **14. Extensiones opcionales**

Solo después del experimento principal:

```text
A. cambiar el modelo embedding 
B. cambiar la métrica
C. cambiar overlap
D. comparar la búsqueda exacta con ANN
```

Cada extensión debe convertirse en un experimento separado.

#### **Puente a la Semana 5**

```text
Semana 4

query -> dense retrieval -> top-k chunks

Semana 5

BM25 --------+
             |
dense -------+-> fusion -> reranking -> contexto -> LLM
```